In [ ]:
"""
================================================================================
stroke-nexus-echo-compare model: Multi-Model & Stacking Ensemble Pipeline
================================================================================
Dataset : Stroke Prediction Dataset
Sampling: SMOTE
Models  : SVM, Gaussian NB, LR, DT, RF, LightGBM, XGBoost, AdaBoost, Extra Trees,
          KNN, CatBoost + 6 Stacking Ensembles (Meta-learner: Logistic Regression)
================================================================================
"""
# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, label_binarize, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LassoCV, LogisticRegression

# Machine Learning Models
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier,
    StackingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    cohen_kappa_score, confusion_matrix, classification_report,
    roc_curve
)

from imblearn.over_sampling import SMOTE

# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
DATA_PATH   = "/kaggle/input/datasets/tofael088/stroke-dataset/stroke.csv"
RANDOM_SEED = 42
TOP_K       = 18
OUTPUT_DIR  = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(RANDOM_SEED)
plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11, "figure.dpi": 100
})

SAMPLING_METHODS = {
    "SMOTE": SMOTE(random_state=RANDOM_SEED)
}

STROKE_CLASSES = ['No Stroke', 'Stroke']

# ─────────────────────────────────────────────────────────────────────────────
# 2. HELPER FUNCTIONS & EVALUATION MATRIX
# ─────────────────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob):
    avg_mode = "weighted"

    is_binary = (y_prob.shape[1] == 2)

    if is_binary:
        auc_val = roc_auc_score(y_true, y_prob[:, 1])
        pr_auc_val = average_precision_score(y_true, y_prob[:, 1])
    else:
        classes_all = np.arange(y_prob.shape[1])
        y_bin = label_binarize(y_true, classes=classes_all)
        valid_cols = y_bin.sum(axis=0) > 0
        if valid_cols.sum() > 1:
            auc_val = roc_auc_score(y_bin[:, valid_cols], y_prob[:, valid_cols], average="weighted")
            pr_auc_val = average_precision_score(y_bin[:, valid_cols], y_prob[:, valid_cols], average="weighted")
        else:
            auc_val, pr_auc_val = 0.5, 0.0

    return dict(
        Accuracy           = accuracy_score(y_true, y_pred),
        Weighted_Precision = precision_score(y_true, y_pred, average=avg_mode, zero_division=0),
        Weighted_Recall    = recall_score(y_true, y_pred, average=avg_mode, zero_division=0),
        Weighted_F1        = f1_score(y_true, y_pred, average=avg_mode, zero_division=0),
        AUC                = auc_val,
        PR_AUC             = pr_auc_val,
        MCC                = matthews_corrcoef(y_true, y_pred),
        Cohen_Kappa        = cohen_kappa_score(y_true, y_pred)
    )

# ─────────────────────────────────────────────────────────────────────────────
# 3. DATA PREPROCESSING, LEAK-FREE TARGET & FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("STEP 1: Loading Data & Securing Splits (Preventing Leaks)")
print("=" * 70)

try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    print("Dataset not found. Ensure you are in Kaggle or update DATA_PATH.")
    exit()

df.columns = df.columns.str.strip()
TARGET = "stroke"

df.drop(columns=["id"], inplace=True, errors='ignore')

if 'smoking_status' in df.columns:
    df['smoking_status'] = df['smoking_status'].fillna('Unknown')

X_raw = df.drop(columns=[TARGET])
y_raw = df[TARGET]

print(f"  --> Target distribution:\n{y_raw.value_counts(normalize=True)*100}")

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_raw, y_raw, test_size=0.20, random_state=RANDOM_SEED, stratify=y_raw
)

# ── Feature Engineering (Medical Context) ──
print("\n  ★  Engineering high-signal physiological metrics...")
bmi_mu = X_train_full["bmi"].mean()

def engineer_features(X_df):
    X = X_df.copy()
    X["bmi_is_missing"] = X["bmi"].isna().astype(int)
    X["bmi_temp"] = X["bmi"].fillna(bmi_mu)

    if "age" in X.columns:
        X["Age_BMI_Ratio"] = X["age"] / (X["bmi_temp"] + 1e-5)
        X["Age_Risk_Flag"] = (X["age"] > 55).astype(int)

        if "hypertension" in X.columns and "heart_disease" in X.columns:
            X["Metabolic_Risk_Score"] = X["hypertension"] + X["heart_disease"] + X["Age_Risk_Flag"]

    if "avg_glucose_level" in X.columns:
        X["Glucose_Danger_Zone"] = (X["avg_glucose_level"] > 140).astype(int)
        X["Glucose_BMI_Interaction"] = X["avg_glucose_level"] * X["bmi_temp"]

    X.drop(columns=["bmi_temp", "Age_Risk_Flag"], inplace=True, errors="ignore")
    return X

X_train_eng = engineer_features(X_train_full)
X_test_eng  = engineer_features(X_test)
new_feats = ["bmi_is_missing", "Age_BMI_Ratio", "Metabolic_Risk_Score", "Glucose_Danger_Zone", "Glucose_BMI_Interaction"]

categorical_cols = X_train_eng.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train_eng.select_dtypes(exclude=['object', 'category']).columns.tolist()

encoder = OneHotEncoder(drop=None, sparse_output=False, handle_unknown='ignore')
encoder.fit(X_train_eng[categorical_cols])

X_train_cat_encoded = pd.DataFrame(
    encoder.transform(X_train_eng[categorical_cols]),
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_train_eng.index
)
X_test_cat_encoded = pd.DataFrame(
    encoder.transform(X_test_eng[categorical_cols]),
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_test_eng.index
)

X_train_eng = pd.concat([X_train_eng[numeric_cols], X_train_cat_encoded], axis=1)
X_test_eng  = pd.concat([X_test_eng[numeric_cols], X_test_cat_encoded], axis=1)

X_train_eng = X_train_eng.astype(float)
X_test_eng  = X_test_eng.astype(float)

feature_names_all = list(X_train_eng.columns)
print(f"  --> Final Shape after feature engineering: {X_train_eng.shape}")

# ─────────────────────────────────────────────────────────────────────────────
print("\nSTEP 2: Safe Data Scaling & Imputation")
# ─────────────────────────────────────────────────────────────────────────────
imputer        = SimpleImputer(strategy="median")
X_train_imp    = imputer.fit_transform(X_train_eng)
X_test_imp     = imputer.transform(X_test_eng)

scaler         = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled  = scaler.transform(X_test_imp)

X_tr_df = pd.DataFrame(X_train_scaled, columns=feature_names_all)
X_ts_df = pd.DataFrame(X_test_scaled,  columns=feature_names_all)

# ─────────────────────────────────────────────────────────────────────────────
print("\nSTEP 3: Ensemble Feature Selection (LASSO + MI)")
# ─────────────────────────────────────────────────────────────────────────────
lasso = LassoCV(cv=5, random_state=RANDOM_SEED, max_iter=10000, n_jobs=-1)
lasso.fit(X_train_scaled, y_train_full)
lasso_coefs = np.abs(lasso.coef_)
lasso_norm  = (lasso_coefs - lasso_coefs.min()) / (lasso_coefs.max() - lasso_coefs.min() + 1e-12)

mi_scores = mutual_info_classif(X_train_scaled, y_train_full, random_state=RANDOM_SEED, n_neighbors=5)
mi_norm = (mi_scores - mi_scores.min()) / (mi_scores.max() - mi_scores.min() + 1e-12)

ensemble_scores = (lasso_norm + mi_norm) / 2.0
feat_score_df = pd.DataFrame({
    "Feature":        feature_names_all,
    "Ensemble_Score": ensemble_scores,
}).sort_values("Ensemble_Score", ascending=False).reset_index(drop=True)

k_features = min(TOP_K, len(feature_names_all))
selected_features = feat_score_df.head(k_features)["Feature"].tolist()

for f in new_feats:
    if f in feature_names_all and f not in selected_features:
        selected_features.append(f)

print(f"  --> Top {len(selected_features)} features selected.")
X_train_sel = X_tr_df[selected_features].values
X_test_sel  = X_ts_df[selected_features].values

# ─────────────────────────────────────────────────────────────────────────────
# 4. MODEL DEFINITIONS & STACKING ENSEMBLES
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 4: Applying SMOTE & Initializing Models")
print("=" * 70)

sampler = SAMPLING_METHODS["SMOTE"]
X_res, y_res = sampler.fit_resample(X_train_sel, np.asarray(y_train_full))
print(f"  --> Training data shape after SMOTE: X={X_res.shape}, y={y_res.shape}")

base_models = {
    "SVM": SVC(probability=True, random_state=RANDOM_SEED, max_iter=5000, cache_size=2000),
    "Gaussian NB": GaussianNB(),
    "Logistic Regression": LogisticRegression(random_state=RANDOM_SEED, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_SEED, max_depth=10),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1),
    "LightGBM": LGBMClassifier(random_state=RANDOM_SEED, verbose=-1, n_jobs=-1),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=RANDOM_SEED, n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(random_state=RANDOM_SEED),
    "Extra Trees": ExtraTreesClassifier(random_state=RANDOM_SEED, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_jobs=-1),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=RANDOM_SEED)
}

meta_lr = LogisticRegression(random_state=RANDOM_SEED, max_iter=1000)

stacking_ensembles = {
    "Stack (RF + XGB)": StackingClassifier(
        estimators=[('rf', base_models['Random Forest']), ('xgb', base_models['XGBoost'])],
        final_estimator=meta_lr, cv=5, n_jobs=-1
    ),
    "Stack (RF + CatBoost)": StackingClassifier(
        estimators=[('rf', base_models['Random Forest']), ('cat', base_models['CatBoost'])],
        final_estimator=meta_lr, cv=5, n_jobs=-1
    ),
    "Stack (RF + ET)": StackingClassifier(
        estimators=[('rf', base_models['Random Forest']), ('et', base_models['Extra Trees'])],
        final_estimator=meta_lr, cv=5, n_jobs=-1
    ),
    "Stack (CatBoost + ET)": StackingClassifier(
        estimators=[('cat', base_models['CatBoost']), ('et', base_models['Extra Trees'])],
        final_estimator=meta_lr, cv=5, n_jobs=-1
    ),
    "Stack (CatBoost + XGB)": StackingClassifier(
        estimators=[('cat', base_models['CatBoost']), ('xgb', base_models['XGBoost'])],
        final_estimator=meta_lr, cv=5, n_jobs=-1
    ),
    "Stack (ET + XGB)": StackingClassifier(
        estimators=[('et', base_models['Extra Trees']), ('xgb', base_models['XGBoost'])],
        final_estimator=meta_lr, cv=5, n_jobs=-1
    )
}

all_models = {**base_models, **stacking_ensembles}

# ─────────────────────────────────────────────────────────────────────────────
# 5. MODEL TRAINING & EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 5: Model Training and Evaluation Pipeline")
print("=" * 70)

results = []
predictions = {}
probabilities = {}

for name, model in all_models.items():
    print(f"  Training {name}...")
    t0 = time.time()

    model.fit(X_res, y_res)

    y_pred = model.predict(X_test_sel)
    y_prob = model.predict_proba(X_test_sel)

    elapsed = time.time() - t0

    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["Model"] = name
    metrics["Time(s)"] = elapsed
    results.append(metrics)

    predictions[name] = y_pred
    probabilities[name] = y_prob

results_df = pd.DataFrame(results)

cols = ["Model", "AUC", "Accuracy", "Weighted_Precision", "Weighted_Recall", "Weighted_F1", "PR_AUC", "MCC", "Time(s)"]
results_df = results_df[cols].sort_values(by="AUC", ascending=False).reset_index(drop=True)

print("\n" + "=" * 90)
print("FINAL PIPELINE SUMMARY (Ranked by AUC)")
print("=" * 90)
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
results_df.to_csv(f"{OUTPUT_DIR}/00_model_metrics_summary.csv", index=False)

# ─────────────────────────────────────────────────────────────────────────────
# 6. VISUALIZATIONS (Confusion Matrices & ROC Curves)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  Generating Evaluation Figures (Saved to {OUTPUT_DIR})...")

n_models = len(all_models)
cols_grid = 4
rows_grid = (n_models + cols_grid - 1) // cols_grid

fig, axes = plt.subplots(rows_grid, cols_grid, figsize=(cols_grid * 4, rows_grid * 3.5))
axes = axes.flatten()

for i, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[i]
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=STROKE_CLASSES, yticklabels=STROKE_CLASSES)
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Confusion Matrices for All Models", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(10, 8))
cmap_base = plt.colormaps.get_cmap("tab20")

for i, name in enumerate(base_models.keys()):
    y_prob = probabilities[name]
    fpr, tpr, _ = roc_curve(y_test, y_prob[:, 1])
    auc_val = roc_auc_score(y_test, y_prob[:, 1])
    ax.plot(fpr, tpr, lw=2, color=cmap_base(i/len(base_models)), label=f"{name} (AUC = {auc_val:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6)
ax.grid(alpha=0.3)
ax.legend(loc="lower right", fontsize=9)
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curves - Base Models")
plt.savefig(f"{OUTPUT_DIR}/02_roc_curves_base_models.png", dpi=150, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(10, 8))
cmap_stack = plt.colormaps.get_cmap("Set1")

for i, name in enumerate(stacking_ensembles.keys()):
    y_prob = probabilities[name]
    fpr, tpr, _ = roc_curve(y_test, y_prob[:, 1])
    auc_val = roc_auc_score(y_test, y_prob[:, 1])
    ax.plot(fpr, tpr, lw=2.5, color=cmap_stack(i/len(stacking_ensembles)), label=f"{name} (AUC = {auc_val:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6)
ax.grid(alpha=0.3)
ax.legend(loc="lower right", fontsize=10)
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curves - Stacking Ensembles")
plt.savefig(f"{OUTPUT_DIR}/03_roc_curves_stacking.png", dpi=150, bbox_inches="tight")
plt.close()

print("\n✓ Pipeline Execution Complete. All outputs saved securely.")

STEP 1: Loading Data & Securing Splits (Preventing Leaks)
  --> Target distribution:
stroke
0    98.195853
1     1.804147
Name: proportion, dtype: float64

  ★  Engineering high-signal physiological metrics...
  --> Final Shape after feature engineering: (34720, 26)

STEP 2: Safe Data Scaling & Imputation

STEP 3: Ensemble Feature Selection (LASSO + MI)
  --> Top 18 features selected.

STEP 4: Applying SMOTE & Initializing Models
  --> Training data shape after SMOTE: X=(68188, 18), y=(68188,)

STEP 5: Model Training and Evaluation Pipeline
  Training SVM...
  Training Gaussian NB...
  Training Logistic Regression...
  Training Decision Tree...
  Training Random Forest...
  Training LightGBM...
  Training XGBoost...
  Training AdaBoost...
  Training Extra Trees...
  Training KNN...
  Training CatBoost...
  Training Stack (RF + XGB)...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:16:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[07:16:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[07:16:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[07:16:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[07:16:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[07:16:58] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.



  Training Stack (RF + CatBoost)...
  Training Stack (RF + ET)...
  Training Stack (CatBoost + ET)...
  Training Stack (CatBoost + XGB)...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:21:58] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[07:22:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[07:22:18] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[07:22:18] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[07:22:18] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[07:22:20] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.



  Training Stack (ET + XGB)...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:23:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[07:23:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[07:23:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[07:23:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[07:23:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[07:23:25] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.




FINAL PIPELINE SUMMARY (Ranked by AUC)
                 Model    AUC  Accuracy  Weighted_Precision  Weighted_Recall  Weighted_F1  PR_AUC     MCC  Time(s)
   Logistic Regression 0.8450    0.7680              0.9777           0.7680       0.8529  0.1056  0.1695   0.7488
              AdaBoost 0.8330    0.7073              0.9777           0.7073       0.8123  0.0889  0.1472   5.8646
           Gaussian NB 0.8115    0.6431              0.9785           0.6431       0.7660  0.0693  0.1360   0.0339
              LightGBM 0.8000    0.9729              0.9669           0.9729       0.9698  0.0643  0.0673   0.7899
 Stack (RF + CatBoost) 0.7953    0.9707              0.9659           0.9707       0.9683  0.0523  0.0384 109.8810
      Stack (RF + XGB) 0.7951    0.9667              0.9662           0.9667       0.9664  0.0531  0.0478  52.4551
 Stack (CatBoost + ET) 0.7922    0.9727              0.9649           0.9727       0.9688  0.0497  0.0126  90.7782
      Stack (ET + XGB) 0.7910    0.9681 